## Prepare SDES Stock From Raw Inputs

**Purpose**
Cleans and harmonizes raw SDES-2018 tables, then builds the aggregated stock table used by downstream calibration and scenario notebooks.

**Primary inputs**
- `data/source/sdes_2018/*.csv`
- `project/input/resources_data.csv`

**Primary outputs**
- `data/derived/building_stock/building_stock_sdes2018_aggregated.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# SDES-2018 Building stock - CONFIDENTIAL


In [ ]:
import os
import pandas as pd
from IPython.display import display

from project.building import reindex_mi


In [ ]:
folder_input = os.path.join('data', 'source', 'sdes_2018')
folder_output = os.path.join('data', 'derived', 'building_stock')
os.makedirs(folder_output, exist_ok=True)


# Main input
Sources: SDES-2018
Status: Confidential


In [ ]:
replace_dict = {r'^(P)$': 'Owner-occupied',
                'Gaz': 'Natural gas',
                'Bois\*': 'Wood fuel',
                'Fioul domestique': 'Oil fuel',
                'Gaz de pétrole liquéfié': 'Oil fuel',
                'Chauffage urbain': 'Heating',
                'MA': 'Single-family',
                'Maison': 'Single-family',
                'AP': 'Multi-family',
                'Appartement': 'Multi-family',
                'LP': 'Privately rented',
                'Autres.*': 'Others',
                'LS': 'Social-housing',
                '.?lectricit.*': 'Electricity',}


## Cleaning names


In [ ]:
name_file = 'comptages_DPE.csv'
stock_buildings = pd.read_csv(os.path.join(folder_input, name_file), sep=',', header=[0], encoding='latin-1',
                        index_col=[0, 1, 2, 3, 4]).squeeze()
index_names = ['Housing type', 'Occupancy status', 'Income tenant', 'Heating energy', 'Energy performance']
stock_buildings.index.set_names(index_names, inplace=True)

stock_buildings = stock_buildings.reset_index().replace(replace_dict, regex=True).set_index(stock_buildings.index.names).squeeze()

display(stock_buildings.head().to_frame().style.format('{:.1f}'))
display(stock_buildings.groupby('Heating energy').sum() / 10**6)
print('Total number of housing in this study {:,.0f}'.format(stock_buildings.sum()))


In [ ]:
stock_buildings.groupby('Energy performance').sum() / 10**6


## Remove gratuity


In [ ]:
stock_buildings = stock_buildings.loc[stock_buildings.index.get_level_values('Occupancy status') != 'G']
display(stock_buildings.head().to_frame().style.format('{:.1f}'))
print('Total number of housing at this point {:,.0f}'.format(stock_buildings.sum()))


In [ ]:
stock_buildings.groupby('Energy performance').sum() / 10**6



# Add owner income as attribute for each building

Using another source of data, we add another level (or attribute) to building stocks: income owner.
Income owner is useful to determine socio-economic parameters like the interest rate or the investment duration.


In [ ]:
display(stock_buildings.groupby('Occupancy status').sum())


## Read data income landlord


In [ ]:
name_file = 'parclocatifprive_post48_revenusPB.csv'
data_income_owner = pd.read_csv(os.path.join(folder_input, name_file), sep=',', header=[0],
                                index_col=[2, 0, 3, 5, 6])
display(data_income_owner.head())


## Cleaning


In [ ]:
index_names = ['Housing type', 'Occupancy status', 'Income tenant', 'Heating energy', 'Energy performance']
data_income_owner.index.set_names(index_names, inplace=True)

data_income_owner.rename(columns={'DECILE_PB': 'Income owner'}, inplace=True)
data_income_owner.reset_index(inplace=True)
data_income_owner.set_index(index_names + ['Income owner'], inplace=True)

data_income_owner = data_income_owner.reset_index().replace(replace_dict, regex=True).set_index(data_income_owner.index.names).squeeze()

data_income_owner = data_income_owner.loc[data_income_owner.index.get_level_values('Income owner') != 'NC']
data_income_owner = data_income_owner.loc[:, 'NB_LOG']

display(data_income_owner.head())
print('\n Total number of housing at this point {:,.0f} - stock with income owner'.format(data_income_owner.sum()))


## Merging data


In [ ]:
print('\n Total number of housing at this point {:,.0f}'.format(stock_buildings.sum()))
# multiplication will remove other value than Landlords (need to be added back later)
share_income_owner = (data_income_owner.unstack('Income owner').T / data_income_owner.unstack('Income owner').sum(axis=1)).T
stock_buildings_landlords = (stock_buildings * reindex_mi(share_income_owner, stock_buildings.index).T).T
stock_buildings_landlords = stock_buildings_landlords.stack().squeeze()

stock_buildings_owners = stock_buildings[stock_buildings.index.get_level_values('Occupancy status') == 'Owner-occupied']
stock_buildings_owners = pd.concat((stock_buildings_owners, pd.Series(stock_buildings_owners.index.get_level_values('Income tenant'), index=stock_buildings_owners.index, name='Income owner')), axis=1)
stock_buildings_owners = stock_buildings_owners.set_index('Income owner', append=True).squeeze()

stock_buildings_social = stock_buildings[stock_buildings.index.get_level_values('Occupancy status') == 'Social-housing']
stock_buildings_social = pd.concat((stock_buildings_social, pd.Series('D10', index=stock_buildings_social.index, name='Income owner')), axis=1)
stock_buildings_social = stock_buildings_social.set_index('Income owner', append=True).squeeze()

stock_buildings = pd.concat((stock_buildings_landlords, stock_buildings_owners, stock_buildings_social))

print(stock_buildings)
print('\n Total number of housing at this point {:,.0f}'.format(stock_buildings.sum()))


# De-aggregate 'Others' to 'Wood fuel', 'Oil fuel' and 'District heating'

Using another source of data, we de-aggregate each rows where Heating energy == 'Others' to 'Wood fuel' and 'Oil fuel'.  
Rate depends on Housing type


## Read data oil fuel and wood fuel


In [ ]:
name_file = 'fuel_oil_wood_2018.xlsx'
data_fuel = pd.read_excel(os.path.join(folder_input, name_file), header=[0], index_col=[1, 0])
display(data_fuel.head(10))

data_fuel.index.set_names(['Heating energy', 'Housing type'], inplace=True)
data_fuel = data_fuel.reset_index().replace(replace_dict, regex=True).set_index(data_fuel.index.names).squeeze()

fuel_list = ['Wood fuel', 'Oil fuel', 'Heating']
data_fuel = data_fuel.loc[data_fuel.index.get_level_values('Heating energy').isin(fuel_list), 'Taux du parc en %']
display(data_fuel.head(10).to_frame().style.format('{:.2f}'))


In [ ]:
## Merging data_fuel with stock_buildings
print('\n Total number of housing at this point {:,.0f}'.format(stock_buildings.sum()))
print(stock_buildings.groupby('Heating energy').sum())

data_fuel = data_fuel.to_frame().pivot_table(columns='Heating energy', index='Housing type').droplevel(None, axis=1)
data_fuel = pd.concat([data_fuel], keys=['Others'], names=['Heating energy'], axis=0)
display(data_fuel.style.format('{:.0%}'))

data_fuel = reindex_mi(data_fuel, stock_buildings.index)

# multiplication will remove other value than Others (need to be added back later)
stock_buildings_others = (stock_buildings * data_fuel.T).T
stock_buildings_others = stock_buildings_others.droplevel('Heating energy', axis=0).stack().squeeze()
stock_buildings_others.dropna(inplace=True)
stock_buildings_others = stock_buildings_others.reorder_levels(stock_buildings.index.names)

stock_buildings = pd.concat((stock_buildings.loc[stock_buildings.index.get_level_values('Heating energy') != 'Others'], stock_buildings_others), axis=0)
display(stock_buildings.groupby('Heating energy').sum())
print('\n Total number of housing at this point {:,.0f}'.format(stock_buildings.sum()))



In [ ]:
stock_buildings.groupby('Energy performance').sum()


# Export results


In [ ]:
print(stock_buildings.to_frame())
print('\n Total number of housings {:,.0f}'.format(stock_buildings.sum()))
stock_buildings.to_csv(os.path.join(folder_output, 'building_stock_sdes2018_aggregated.csv'))
